In [ ]:
import glob
import os
from pathlib import Path
import numpy as np
import polars as pl
import polars.selectors as cs
!pip install fastdtw
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.markers import MarkerStyle
import matplotlib.patches as patches
import seaborn as sns
from IPython.display import Image, display


In [ ]:
# config
PARENTS_PATH = r"/kaggle/input/competitions/rogii-wellbore-geology-prediction"


In [ ]:
def display_basename(path):
    return os.path.basename(path)

def get_filepath(traintest:str, filename:str):
    return os.path.join(PARENTS_PATH, traintest, filename)

def check_null_ratios(df: pl.DataFrame) -> pl.DataFrame:
    return (df.null_count() / len(df))

def describe_with_null_ratio(df: pl.DataFrame) -> pl.DataFrame:
    desc = df.describe().filter(pl.col("statistic") != "null_count")
    # get folumnsexcept for "statistic" column
    target_cols = desc.columns[1:]
    null_ratio_row = df.select([
        pl.lit("null_ratio").alias("statistic"),
        *[(pl.col(c).null_count() / len(df)).cast(desc.schema[c]).alias(c) for c in target_cols]
    ])
    return pl.concat([desc, null_ratio_row])

## Preview train dataset

In [ ]:
train_csvfiles = []
train_csvfiles += glob.glob(os.path.join(PARENTS_PATH, 'train', '*.csv'))
print("num of csv files in train dir:", len(train_csvfiles))

test_csvfiles = []
test_csvfiles += glob.glob(os.path.join(PARENTS_PATH, 'test', '*.csv'))
print("num of csv files in test dir:", len(test_csvfiles))

train_pngfiles = []
train_pngfiles += glob.glob(os.path.join(PARENTS_PATH, 'train', '*.png'))
print("num of png files in train dir:", len(train_pngfiles))

In [ ]:
print(display_basename(train_pngfiles[0]))
display(Image(train_pngfiles[0]))
for file in train_csvfiles:
    if Path(train_pngfiles[0]).stem in file:
        print(f"\n==={display_basename(file)}===")
        train_df = pl.read_csv(file)
        display(train_df)
        print("\n///statistics///")
        display(describe_with_null_ratio(train_df))


In [ ]:
pl.read_csv(get_filepath("train", "cd7f1687__horizontal_well.csv"	))

## Preview test data set

In [ ]:
for file in test_csvfiles:
    print(f"\n==={display_basename(file)}===")
    test_df = pl.read_csv(file)
    display(test_df)
    print("\n///statistics///")
    display(describe_with_null_ratio(test_df))

# Analysis of horizontal files

In [ ]:
# Threshold for the change in TVT to be considered flat
FLAT_THRESHOLD = 0.5 

# focus on horizontal files
horizontal_files = [f for f in train_csvfiles if "typewell" not in f]

summary_records_horizontal = []
for path in horizontal_files:
    df = pl.read_csv(path, columns=["MD", "GR", "TVT"])
    df = df.with_columns(pl.col("GR").cast(pl.Float64))
    
    gr_std = df["GR"].drop_nulls().std()
    gr_null_ratio = df["GR"].null_count() / len(df)
    tvt_diff_std = df["TVT"].drop_nulls().diff().std()
    
    # Calculate the difference in TVT and filter only the rows whose absolute value is less than or equal to the threshold.
    df = df.with_columns(pl.col("TVT").diff().abs().alias("tvt_diff_abs"))
    flat_df = df.filter(pl.col("tvt_diff_abs") < FLAT_THRESHOLD)
    
    # Conditional branching is used to handle cases where there are no files with flat sections.
    if len(flat_df.drop_nulls(subset=["GR"])) > 0:
        flat_gr_std = flat_df["GR"].std()
    else:
        flat_gr_std = np.nan # If there is no flat section, the result is NaN.

    summary_records_horizontal.append({
        "filename": os.path.basename(path),
        "path": path,
        "gr_std": gr_std,
        "gr_null_ratio": gr_null_ratio,
        "tvt_diff_std": tvt_diff_std,
        "flat_gr_std": flat_gr_std,
        "total_rows": len(df)
    })

summary_df_horizontal = pl.DataFrame(summary_records_horizontal)
display(summary_df_horizontal.head())

## Correlation analysis of GR and TVT

In [ ]:
# 1. Extreme wells (top 5 with the largest noise std_GR)
extreme_wells_horizontal = summary_df_horizontal.sort("gr_std", descending=True).head(5)

# 2. Average wells (5 wells with noise std_GR close to the median)
median_val = summary_df_horizontal["gr_std"].median()
average_wells_horizontal = (
    summary_df_horizontal
    .with_columns((pl.col("gr_std") - median_val).abs().alias("diff_from_median"))
    .sort("diff_from_median")
    .head(5)
)

print("Extreme Wells (High Noise):\n", extreme_wells_horizontal["filename"].to_list())
print("\nAverage Wells (Median Noise):\n", average_wells_horizontal["filename"].to_list())


def plot_well_comparison(wells_df, title):
    fig, axes = plt.subplots(len(wells_df), 1, figsize=(15, 4 * len(wells_df)), sharex=False)
    if len(wells_df) == 1: axes = [axes]

    for i, row in enumerate(wells_df.to_dicts()):
        df = pl.read_csv(row['path'])
        ax = axes[i]
        df = df.with_columns(
            pl.col("GR").cast(pl.Float64)
        )

        # Left axis: GR (observed value)
        ax.plot(df["MD"], df["GR"], color="blue", alpha=0.6, label="GR (Observation)")
        ax.set_ylabel("Gamma Ray", color="blue")
        ax.set_ylim(0,300)
        ax.set_xlim(10000,20000)

        # Right axis: TVT (True value/Stratum)
        ax2 = ax.twinx()
        ax2.plot(df["MD"], df["TVT"], color="red", linewidth=2, label="TVT (State)")
        ax2.set_ylabel("TVT", color="red")
        ax2.set_ylim(10000,13000)
        ax.set_xlim(10000,20000)

        ax.set_title(f"Well: {row['filename']} | GR Std: {row['gr_std']:.2f} | Null: {row['gr_null_ratio']:.2%}")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.suptitle(title, fontsize=20, y=1.02)
    plt.show()

# execution
plot_well_comparison(average_wells_horizontal, "Average Wells (Normal Noise)")
plot_well_comparison(extreme_wells_horizontal, "Extreme Wells (High Noise/Anomalies)")

#### note

> Pattern 1: TVT changes (increases/decreases), and GR also fluctuates significantly.
  - **Waveform State:** TVT has an upward (or downward) slope, and GR fluctuates accordingly, rising and falling in waves.
  - **Geological Assumption:** This represents a state where the drill is diagonally penetrating multiple different rock layers.
  - **Explanation:** This occurs in the early stages of drilling (called the curve building, the section where the drill changes direction from vertical to horizontal) or when the drill is advancing diagonally relative to the dip of the rock layers. Because new rock layers are entered one after another with each change in depth (TVT), GR fluctuates violently along the GR profile of the type well (vertical well).
  - **In the Data Provided:** The "early stages of MD" are exactly this state. Since TVT is increasing as the drill crosses the rock layers, a characteristic waveform is likely appearing in the GR.

> Pattern 2: TVT is flat (constant) and GR is stable
  - **Waveform State:** The slope of TVT is zero (horizontally horizontal), and GR also moves steadily within a specific numerical range.
  - **Geological Assumption:** This is a state where the drill is drilling perfectly horizontally right through the center of the target stratum (target zone).
  - **Explanation:** This is an ideal state where the slope of the stratum and the angle of the drill's advancement are perfectly matched. Because the drill is moving through the same rock continuously, neither TVT (relative depth of stratum) nor GR (rock composition) changes.
  - **In this data:** The baseline "after the MD has advanced to a certain extent" corresponds to this state.

> Pattern 3: TVT is flat (constant), but GR spikes locally (suddenly changes)
  - **Waveform State:** TVT moves flat, but only GR suddenly jumps up and then returns to normal.

  - **Geological Assumption:** Either "grazing the boundary of the rock layer" or **"localized anomaly (fault/noise)."**

  - **Explanation:** Even if the drill is intended to move horizontally (constant TVT), if the rock layer is locally undulating, it may temporarily excavate "rock from the adjacent layer" (boundary bounce). Alternatively, it could be a simple sensor error, or the accidental presence of another component within that layer (lens-like structure).

  - **In this data:** The "huge spikes in GR occurring despite stable TVT," seen in extreme wells, are highly likely to be this phenomenon.

> Pattern 4: TVT is changing, but GR is stable.

  - **Waveform State:** TVT is increasing and decreasing, but GR remains at a constant value.

  - **Geological Assumption:** This describes "drilling diagonally through a single, very thick, homogeneous rock layer."

  - **Explanation:** For example, in a geological formation where the same sandstone continues for hundreds of feet, the GR will not react because the rock composition does not change regardless of how much the TVT changes. In this section, it becomes very difficult to determine "which TVT you are currently in" by looking only at the GR.

## Analysis of noise and null pattern

### Variance in Flat Intervals

In [ ]:
print("=== Standard deviation of GR in flat sections ===")
flat_stats = summary_df_horizontal.select(
    pl.col("flat_gr_std").mean().alias("Mean"),
    pl.col("flat_gr_std").std().alias("Std"),
    pl.col("flat_gr_std").max().alias("Max"),
    pl.col("flat_gr_std").min().alias("Min")
)
print(flat_stats)

#### note
- Even in the most stable wells, the deviation reaches $\pm 6.5$, and in rougher wells, it reaches $\pm 34.2$.

- This confirms that the problem of "multimodality (multiple strata having similar GR values)" becomes even more complicated. It is impossible to determine from a single point whether a "GR shift of 15" is "noise" or "entering an adjacent strata."

### TVT rate of change and dip

In [ ]:
# Group the results into 10 lists for plotting (5 average results + 5 extreme results).
target_wells = average_wells_horizontal.to_dicts() + extreme_wells_horizontal.to_dicts()

fig, axes = plt.subplots(5, 2, figsize=(15, 15))
axes = axes.flatten()

X_LIM = (-10, 10)  # TVT change per step
Y_LIM = (0, 1000)  # Frequency (number of lines)

for i, row in enumerate(target_wells):
    df = pl.read_csv(row['path']).with_columns(pl.col("TVT").cast(pl.Float64))
    # Calculate the difference between TVTs, remove nulls, and convert to a NumPy array.
    tvt_diffs = df["TVT"].diff().drop_nulls().to_numpy()
    
    ax = axes[i]
    sns.histplot(tvt_diffs, bins=50, ax=ax, kde=True, color="teal")
    
    ax.set_xlim(X_LIM)
    ax.set_ylim(Y_LIM)
    
    well_type = "Average" if i < 5 else "Extreme"
    ax.set_title(f"[{well_type}] {row['filename'][:20]}...", fontsize=10)
    ax.set_xlabel("Delta TVT")
    ax.set_ylabel("Count")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle("Distribution of Delta TVT (Dip analysis)", fontsize=16, y=1.02)
plt.show()

#### note
- For most steps, $\Delta \text{TVT}$ is close to 0 (moving horizontally) or very small. 
- This is evidence that the data reflects the mechanical constraint that "a drill, a giant iron tube, cannot bend up or down at a sharp angle."

### Visualization of missing value patterns

In [ ]:
fig, axes = plt.subplots(10, 1, figsize=(12, 10), sharex=False) # Set sharex to False to adjust to the range of each well.

for i, row in enumerate(target_wells):
    df = pl.read_csv(row['path']).with_columns([
        pl.col("MD").cast(pl.Float64),
        pl.col("GR").cast(pl.Float64)
    ]).sort("MD")
    
    # Obtain the effective range of the MD for each well.
    min_md, max_md = df["MD"].min(), df["MD"].max()
    
    ax = axes[i]
    # Set the background to a section where no data exists (gray).
    ax.set_facecolor('#f4f4f4')
    
    # 1. Fill the valid data range with black (indicating data).
    ax.add_patch(plt.Rectangle((min_md, 0), max_md - min_md, 1, color='black', zorder=1))
    
    # 2. The sections where GR is Null are highlighted in yellow.
    # Treat isnull() as a boolean value and paint the interval where it is True.
    null_mask = df["GR"].is_null().to_numpy()
    md_vals = df["MD"].to_numpy()
    
    # Fill in areas where Null is repeated as a block.
    ax.fill_between(md_vals, 0, 1, where=null_mask, color='yellow', step="mid", zorder=2)
    
    ax.set_xlim(min_md - 50, max_md + 50)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    
    well_type = "Average" if i < 5 else "Extreme"
    ax.set_ylabel(f"{well_type}\n{row['filename'][:8]}", rotation=0, labelpad=40, va='center', fontsize=9)

plt.tight_layout()
plt.suptitle("GR Data Availability (Corrected): Yellow=Null, Black=Valid", fontsize=16, y=1.02)
plt.xlabel("Measured Depth (MD)")
plt.show()

#### note
- Some of the extreme wells are almost completely null.

- This is not a matter of temporary sensor errors, but rather a structural omission (MNAR: Missing Not At Random) where gamma-ray logging is intentionally not performed or is lost in certain sections (e.g., specific geological formations or the final phase of drilling).

# Analysis of typewell files

In [ ]:
# focus on horizontal files
typewell_files = [f for f in train_csvfiles if "horizontal" not in f]

summary_records_typewell = []
for path in typewell_files:
    # calculate of statistics
    df = pl.read_csv(path, columns=["Geology", "GR", "TVT"])

    # GR
    gr_std = df["GR"].drop_nulls().cast(pl.Float64).std()
    gr_null_ratio = df["GR"].null_count() / len(df)

    # changes of TVT （for system noise Q）
    # look at the standard deviation of the change in TVT for each step using the data up to the PS point.
    tvt_diff_std = df["TVT"].drop_nulls().diff().std()

    summary_records_typewell.append({
        "filename": os.path.basename(path),
        "path": path,
        "gr_std": gr_std,
        "gr_null_ratio": gr_null_ratio,
        "tvt_diff_std": tvt_diff_std,
        "total_rows": len(df)
    })

summary_df_typewell = pl.DataFrame(summary_records_typewell)
display(summary_df_typewell.head())

In [ ]:
# 1. Extreme wells (top 5 with the largest noise std_GR)
extreme_wells_typewell = summary_df_typewell.sort("gr_std", descending=True).head(5)

# 2. Average wells (5 wells with noise std_GR close to the median)
median_val = summary_df_typewell["gr_std"].median()
average_wells_typewell = (
    summary_df_typewell
    .with_columns((pl.col("gr_std") - median_val).abs().alias("diff_from_median"))
    .sort("diff_from_median")
    .head(5)
)

print("Extreme Wells (High Noise):\n", extreme_wells_typewell["filename"].to_list())
print("\nAverage Wells (Median Noise):\n", average_wells_typewell["filename"].to_list())

def scatter_plot_wells(wells_df, title):
    all_dfs = []
    for row in wells_df.to_dicts():
        df = pl.read_csv(row['path'])
        df = df.with_columns(pl.col("GR").cast(pl.Float64))
        df = df.with_columns(pl.lit(row['filename']).alias("source_well"))
        all_dfs.append(df)
    
    combined_df = pl.concat(all_dfs).to_pandas()
    
    unique_geologies = sorted(combined_df["Geology"].dropna().unique())
    
    if len(unique_geologies) <= 10:
        palette = sns.color_palette("tab10", len(unique_geologies))
    else:
        palette = sns.color_palette("tab20", len(unique_geologies))
        
    color_map = dict(zip(unique_geologies, palette))
    
    plt.figure(figsize=(12, 9))    
    sns.scatterplot(
        data=combined_df, 
        x="GR", 
        y="TVT", 
        hue="Geology", 
        palette=color_map,
        alpha=0.9, 
        s=30
    )
    
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.xlim(10,450)
    plt.ylim(12500,10000)
    plt.xlabel("Gamma Ray (GR)")
    plt.ylabel("True Vertical Thickness (TVT)")
    plt.grid(True, linestyle='--', alpha=0.3)
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, title="Geology", fontsize='large')
    plt.tight_layout()
    plt.show()

# execution
scatter_plot_wells(average_wells_typewell, "Average Wells (Normal Noise)")
scatter_plot_wells(extreme_wells_typewell, "Extreme Wells (High Noise/Anomalies)")

### note
- Differences in Target Depth and Stratigraphic Diversity:

  - Average Wells: TVT is widely distributed from approximately 10300 to 12300, divided into two large layered blocks. There are also 13 different strata, indicating an area where "drilling is being conducted that traverses or moves between deep, diverse strata."

  - Extreme Wells: TVT is concentrated in a very narrow, shallow range from approximately 10500 to 11250. There are only 6 different strata.

- Identification of the "Giant GR Spike":

  - The "intense noise with GR exceeding 300-400" observed in the Extreme Wells plot on the horizontal wells has been completely explained by this scatter plot.

  - This is not a sensor error or anomaly (noise), but a geological fact: the "extremely high radiation levels (GR)" inherent in certain geological formations such as ANCC and EGFDU/EGFDL. Extreme Wells drills near these "special layers where the GR spikes extremely," which explains the drastic waveforms even in the horizontal well data.

# Other analysis

## Calculation of observational noise by comparing vertical and horizontal wells.

In [ ]:
def calculate_r_from_well_comparison(
    summary_df_horizontal, summary_df_typewell
):
  
    def calculate_gr_diff(h_path, t_path):
        df_h = (
            pl.read_csv(h_path)
            .select(["TVT", "GR"])
            .rename({"GR": "GR_h"})
            .cast(pl.Float32)
        )
        df_t = (
            pl.read_csv(t_path)
            .select(["TVT", "GR"])
            .rename({"GR": "GR_t"})
            .cast(pl.Float32)
        )

        combined = df_h.join(df_t, on="TVT", how="inner")
        combined = combined.select(["TVT", "GR_h", "GR_t"]).drop_nulls()

        return combined.with_columns(
            (pl.col("GR_h") - pl.col("GR_t")).alias("GR_diff")
        )

    all_diffs = []

    for h_path in summary_df_horizontal["path"]:
        base_name = os.path.basename(h_path).replace(
            "_horizontal_well.csv", ""
        )
        t_path = [p for p in summary_df_typewell["path"] if base_name in p]

        if t_path:
            diff_df = calculate_gr_diff(h_path, t_path[0])
            diff_df = diff_df.with_columns(
                [
                    pl.lit(os.path.basename(h_path)).alias("h_path"),
                    pl.lit(os.path.basename(t_path[0])).alias("t_path"),
                ]
            )
            all_diffs.append(diff_df)

    if not all_diffs:
        print("No matching well pairs were found.")
        return None

    final_df = pl.concat(all_diffs).to_pandas()

    all_residuals = final_df["GR_diff"].values

    R_proposed = np.var(all_residuals)

    print(f"Total number of data points (number of TVTs that matched perfectly): {len(all_residuals)}")
    print(f"Average value of GR difference (closer to 0 indicates higher consistency): {np.mean(all_residuals):.4f}")
    print(f"variance: {R_proposed:.4f}")
    print(f"Standard deviation of error (Std): {np.sqrt(R_proposed):.4f} API")

    return final_df, R_proposed

final_df, R_best = calculate_r_from_well_comparison(summary_df_horizontal, summary_df_typewell)

plt.figure(figsize=(10, 6))
sns.histplot(final_df['GR_diff'], kde=True, bins=50)
plt.title("Distribution of GR Difference (Horizontal - Typewell)")
plt.xlabel("Delta GR")
plt.grid(True, alpha=0.3) 
plt.show()

summary_table = final_df['GR_diff'].describe().to_frame().T
print("\n=== Summary Table of GR Difference ===")
display(summary_table)

stats = final_df['GR_diff'].describe()
quantiles = {
    "min": stats['min'],
    "25%": stats['25%'],
    "50%": stats['50%'],
    "75%": stats['75%'],
    "max": stats['max']
}

target_rows = []
for label, value in quantiles.items():
    idx = (final_df['GR_diff'] - value).abs().idxmin()
    target_row = final_df.loc[[idx]]
    target_rows.append(target_row)
import pandas as pd
target_rows_df = pd.concat(target_rows)    
print(f"Data for the relevant row:")
display(target_rows_df)


## DTW Based Waveform Matching (Horizontal vs Typewell)

In [ ]:
def analyze_dtw_matching(horizontal_path, typewell_path):
    """
    Analyze the structural similarity between horizontal well GR and typewell GR using DTW.
    """
    # Load and clean horizontal well data (using data before PS point)
    # Assume we filter rows where TVT is known or before a specific MD threshold
    h_df = pl.read_csv(horizontal_path).with_columns(pl.col("GR").cast(pl.Float64)).drop_nulls(subset=["GR"])
    
    # Load typewell data
    t_df = pl.read_csv(typewell_path).with_columns(pl.col("GR").cast(pl.Float64)).drop_nulls(subset=["GR"])
    
    # Extract GR sequences and normalize them for fair distance comparison
    h_gr = h_df["GR"].to_numpy()
    t_gr = t_df["GR"].to_numpy()
    
    h_gr_norm = (h_gr - np.mean(h_gr)) / np.std(h_gr)
    t_gr_norm = (t_gr - np.mean(t_gr)) / np.std(t_gr)
    # Reshape 1D arrays to 2D arrays with shape (N, 1) to make them compatible with fastdtw/scipy distance metrics
    h_gr_norm = h_gr_norm.reshape(-1, 1)
    t_gr_norm = t_gr_norm.reshape(-1, 1)

    # Compute Dynamic Time Warping distance and path
    distance, path = fastdtw(h_gr_norm, t_gr_norm, dist=euclidean)
    print(f"Num of data points: horizontal file:{h_gr_norm.shape}, typewell file:{t_gr_norm.shape},")
    print(f"DTW Distance between wells: {distance:.2f}")
    
    # Plot the alignment results
    plt.figure(figsize=(12, 6))
    
    # Visualizing the alignment paths (showing every 20th match to prevent crowding)
    for h_idx, t_idx in path[::20]:
        plt.plot([h_df["MD"][h_idx], t_df["TVT"][t_idx]], [h_gr[h_idx], t_gr[t_idx]], 
                 color="gray", alpha=0.3, linestyle="--")
                 
    plt.plot(h_df["MD"], h_gr, label="Horizontal Well (vs MD)", color="blue", alpha=0.8)
    plt.plot(t_df["TVT"], t_gr, label="Typewell (vs TVT)", color="red", alpha=0.8)
    
    plt.title(f"{os.path.basename(horizontal_path)}_Waveform Alignment via DTW (GR Pattern Matching)")
    plt.xlabel("Depth Scale (MD for Horizontal, TVT for Typewell)")
    plt.ylabel("Gamma Ray (GR)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# 1. Average wells
print("\n=== Average Wells ===\n")
for i, h in enumerate(average_wells_horizontal['path']):
  if i == 6:
    break
  for t in summary_df_typewell['path']:
    if os.path.basename(h).strip('__horizontal_well.csv') == os.path.basename(t).strip('__typewell.csv'):
      print(os.path.basename(h)," & ", os.path.basename(t))
      analyze_dtw_matching(h, t)
      break

# 2. Extreme wells
print("\n\n\n=== Extreme Wells ===\n")
for i, h in enumerate(extreme_wells_horizontal['path']):
  if i == 6:
    break
  for t in summary_df_typewell['path']:
    if os.path.basename(h).strip('__horizontal_well.csv') == os.path.basename(t).strip('__typewell.csv'):
      print(os.path.basename(h), os.path.basename(t))
      analyze_dtw_matching(h, t)
      break

#### note
【A】 Characteristics of the average data set (e03b45fd, 463c38f2, 2ac7cb6b, cd7f1687)

  - Strong horizontal stretching: The gray dotted line fanns out from a specific Typewell depth (TVT) towards a very wide horizontal section (MD). This indicates that the drill successfully lodged in the target layer (target zone) and was able to stably drill horizontally over long distances within that layer.

  - High degree of waveform pattern agreement: The GR fluctuation range and the rhythm of the fine peaks and valleys in the blue (Horizontal) and red (Typewell) lines match relatively well, indicating minimal twisting in the DTW path.

  - Local deviation in absolute values: While the absolute deviation of GR values ​​is not zero, it falls within the range of natural phase changes (gradual changes in rock type) due to the distance between wells.

【B】 Characteristics of the highly variable data set (19137a89, 00e12e8b, 493b5b31, 0dc5e64d)

- Severe GR spikes and discontinuities: Sharp spikes exceeding 150-200 frequently occur in the blue waveform (horizontal). This strongly suggests that the drill is not confined to a single stratum, but is rapidly moving up and down between high and low GR (porpoising), or passing through faults.

- Distortion in DTW matching: Gray dotted lines intersect, and in some places, a "downward slope (stratum inversion or calculation pullback)" occurs. While the shapes are not similar, there are scattered areas where the algorithm is forcibly connecting the "best-case scenario" through "noise absorption."

- Extreme absolute value discrepancies: Numerous locations exist where the GR values ​​differ by more than 50 at both ends of the dotted line. This is not so much sensor noise, but rather evidence that the geological structure is extremely complex, and that Typewell data alone cannot adequately represent the rock type around Horizontal.

## Geometric relationship between XYZ coordinates and TVT, and correlation between direction of propagation and stratum inclination.

In [ ]:

def plot_trajectory_vs_tvt_geometry(horizontal_path):
    """
    Analyze the correlation between 3D trajectory (X, Y, Z) and actual TVT changes
    to extract geological formation dip characteristics.
    """
    # Load and ensure necessary columns are cast to Float64
    df = pl.read_csv(horizontal_path).with_columns([
        pl.col("MD").cast(pl.Float64),
        pl.col("X").cast(pl.Float64),
        pl.col("Y").cast(pl.Float64),
        pl.col("Z").cast(pl.Float64),
        pl.col("TVT").cast(pl.Float64)
    ]).sort("MD")
    
    # Calculate step-by-step changes (differences) for each coordinate
    # Note: Z represents the true vertical depth below sea level
    df = df.with_columns([
        pl.col("X").diff().alias("delta_x"),
        pl.col("Y").diff().alias("delta_y"),
        pl.col("Z").diff().abs().alias("delta_z"), # Absolute vertical movement
        pl.col("TVT").diff().alias("delta_tvt")
    ])
    
    # Calculate the drilling Azimuth (direction in horizontal plane) using arctan2(dX, dY)
    # Convert radians to degrees (0 to 360 degrees)
    df = df.with_columns(
        (((pl.arctan2("delta_x", "delta_y") * 180 / np.pi) + 360) % 360).alias("calculated_azimuth")
    )
    
    # Drop rows containing NaNs in calculated metrics
    pdf = df.drop_nulls(subset=["delta_z", "delta_tvt", "calculated_azimuth"]).to_pandas()
    
    # Create correlation plots
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: True Vertical Depth change (ΔZ) vs Actual Thickness change (ΔTVT)
    # If the geological layer is completely flat, ΔTVT should equal ΔZ.
    sns.scatterplot(data=pdf, x="delta_z", y="delta_tvt", alpha=0.5, ax=axes[0], color="purple")
    
    # Draw reference line (y = x) representing perfectly flat layers
    min_val = min(pdf["delta_z"].min(), pdf["delta_tvt"].min())
    max_val = max(pdf["delta_z"].max(), pdf["delta_tvt"].max())
    axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', label="Perfect Flat Layer Line (ΔTVT = ΔZ)")
    
    axes[0].set_title(f"{os.path.basename(horizontal_path)}\nActual ΔTVT vs True Vertical Depth Change (ΔZ)")
    axes[0].set_xlabel("Vertical Depth Change (ΔZ)")
    axes[0].set_ylabel("Actual Stratum Change (ΔTVT)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    # Plot 2: Discrepancy (Formation Dip Effect) vs Calculated Drilling Azimuth
    # Discrepancy = ΔTVT - ΔZ. If positive/negative trends appear at specific azimuths,
    # it indicates a systematic directional geological dip.
    pdf["dip_discrepancy"] = pdf["delta_tvt"] - pdf["delta_z"]
    
    # Create a scatter plot colored by the step index to see trends along the wellpath
    scatter = axes[1].scatter(pdf["calculated_azimuth"], pdf["dip_discrepancy"], 
                             c=pdf["MD"], cmap="viridis", alpha=0.6)
    cbar = fig.colorbar(scatter, ax=axes[1])
    cbar.set_label("Measured Depth (MD)")
    
    axes[1].set_title(f"{os.path.basename(horizontal_path)}\nGeological Dip Discrepancy vs Calculated Azimuth")
    axes[1].set_xlabel("Calculated Drilling Azimuth (degrees)")
    axes[1].set_ylabel("Discrepancy (ΔTVT - ΔZ)")
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


# 1. Average wells
print("\n=== Average Wells ===\n")
for i, h in enumerate(average_wells_horizontal['path']):
  if i == 6:
    break
  for t in summary_df_typewell['path']:
    if os.path.basename(h).strip('__horizontal_well.csv') == os.path.basename(t).strip('__typewell.csv'):
      print(os.path.basename(h)," & ", os.path.basename(t))
      plot_trajectory_vs_tvt_geometry(h)
      break

# 2. Extreme wells
print("\n\n\n=== Extreme Wells ===\n")
for i, h in enumerate(extreme_wells_horizontal['path']):
  if i == 6:
    break
  for t in summary_df_typewell['path']:
    if os.path.basename(h).strip('__horizontal_well.csv') == os.path.basename(t).strip('__typewell.csv'):
      print(os.path.basename(h), os.path.basename(t))
      plot_trajectory_vs_tvt_geometry(h)
      break

#### note
Results (Plot 1 ΔZ vs ΔTVT)

The basic principle of this plot is that "$\Delta Z$ is the vertical movement of the drill (depth from sea level)" and "$\Delta \text{TVT}$ is the movement relative to the thickness of the rock layer."

- **The plot deviates above and below the red dotted line where y=x**

- This indicates that the rock layer itself is tilted (has a dip), not the drilling method.

- If the rock layer were perfectly horizontal, whether the drill is drilled at an angle or vertically, "if the drill moves 1m vertically, it will also pass through 1m of the rock layer," so it will always lie on $y=x$. Deviation from this means that the rock layer is either "rising up" or "subducting" relative to the direction of the drill's movement, resulting in an increase or decrease in the apparent thickness change of the rock layer ($\Delta \text{TVT}$).

- **There is a section where $\Delta \text{TVT}$ is shifted above the red dotted line only at a specific $\Delta Z$.**

- **Meaning:** This means that in that section, "the slope of the rock layer is locally steeper (or a small fault has been crossed)."

- If $\Delta \text{TVT}$ is larger while $\Delta Z$ is the same, it means that "a lot of rock has been passed through even though only a little has been drilled down." This is a sign that the drill has penetrated a locally undulating (folded) section of the rock layer.

- **$\Delta Z$ and $\Delta \text{TVT}$ are near 0.0.**

- This means that "the drill is drilling almost parallel (horizontally) to the rock layer (horizontal drilling)." This is not the starting point.

- **Explanation:** This value is "the amount of change per step ($\Delta$)." When $\Delta Z \approx 0$ and $\Delta \text{TVT} \approx 0$, it means that there is no movement up or down, and the drill is not entering a different geological layer. In other words, it is a stable section that "cleanly penetrates horizontally through the center of the target geological layer (Sweet Spot)." The majority of horizontal well data is accumulated here.
- **When $\Delta Z$ and $\Delta \text{TVT}$ are near 0.0, as $\Delta Z$ advances, $\Delta \text{TVT}$ will move in the negative direction of 0.0 rather than along the red dotted line.**
- This signifies a very important geological phenomenon: "Even though the drill is drilling downwards, it is penetrating towards the upper part of the geological layer (the ceiling side/younger layer)."
- When $\Delta Z$ advances (becomes positive) = the drill is going downwards. Normally, since it enters an older geological layer, $\Delta \text{TVT}$ will be positive. However, a negative value for $\Delta \text{TVT}$ means that "the entire rock layer is sinking downwards at a steeper angle than the drill, in the direction of the drill's movement." Even though the drill is pointing downwards, the rock layer is pointing even further downwards, so relatively speaking, it's as if the drill has "broken through the ceiling and emerged into the upper layer."


Results (Plot 2: Azimuth vs. Discrepancy)

The basic principle of this plot is that "the X-axis represents the compass direction (0 = North, 90 = East)" and "the Y-axis represents the discrepancy between geometric calculations and actual changes in the geological layers."

- **Plots converge at specific positions along the azimuth angle (x-axis) of the drill's direction of travel.**

- This means the drill is drilling in a straight line towards a constant direction (e.g., due east) without curving along the way.

- **Plots converge at two specific positions along the azimuth angle (x-axis) of the drill's direction of travel (0 degrees and 360 degrees).**

- This means the drill is drilling in a straight line towards true north.

- 0 degrees and 360 degrees are exactly the same direction, "North." When a drill is aiming due north, a slight deviation to the right is recorded as "1 degree," and a deviation to the left is recorded as "359 degrees." Therefore, on the graph, data is split and aggregated at both ends (0 and 360 degrees) (a phenomenon called "wrap-around").

- **A case is observed where the plots are clustered in the x-axis range of 120 to 150 degrees (y-axis values ​​are around 0.0).**

- This means that the drill is proceeding southeast (120-150 degrees), and the rock layer is perfectly horizontal (zero slope) in that direction.

- A Y-axis deviation of around 0.0 is the same as "lying on the red dotted line of plot 1." In other words, as long as the drill proceeds in this 120-150 degree direction, it is not affected at all by the slope of the rock layer.

- **The plots are scattered with x-axis values ​​ranging from about 30.**

- This means that the drill is not moving in a straight line, but is "wobbling (swerving) from side to side, or the trajectory is intentionally curved."

- These marks indicate that when the drill tip was about to stray from the target rock layer, the geologist made slight adjustments to the trajectory, saying things like, "A little to the right, then a little to the left" (steering marks), or that the drill was excavated in a gentle curve to avoid the terrain.